# Bubble-tea revenue: link relative seasonal indices

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Link Relative Method** · a worked example from the StatNex teaching guide

**The story.** A bubble-tea chain has its quarterly revenue from 2020 to 2024, in thousands of dollars. Summer (Q3) is the best quarter in every year, and the business grew every year. This method compares each quarter with the one before it, which takes the growth out before measuring the seasons.

**The question.** After allowing for growth, what are the seasonal indices, and what does revenue look like with the season removed?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `08_time_series/link_relative.csv` (20 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Year / period,Sales ($000)
2020-1Q,792.0
2020-2Q,1132.0
2020-3Q,1359.0
2020-4Q,852.0
2021-1Q,915.0
2021-2Q,1216.0
2021-3Q,1503.0
2021-4Q,960.0
2022-1Q,1005.0
2022-2Q,1385.0
2022-3Q,1732.0
2022-4Q,1083.0
2023-1Q,1075.0
2023-2Q,1600.0
2023-3Q,1801.0
2023-4Q,1140.0
2024-1Q,1184.0
2024-2Q,1522.0
2024-3Q,1861.0
2024-4Q,1247.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

## The data

In [ ]:
# Every value must be positive: the method works in ratios to the previous period.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
# One column in time order, cut into years of periods observations: one row per period,
# one column per year. The number of values must be a whole number of years.
values = data["Sales ($000)"].to_numpy(float)
periods = 4
n_years = len(values) // periods
table = values.reshape(n_years, periods).T
# The first year only labels the output; it does not change any number.
years = [2020 + c for c in range(n_years)]
series = table.T.ravel()  # back in time order, year by year

## Link relatives

In [ ]:
# Each value divided by the one before it. Its ratio to the previous period cancels
# whatever level the series has reached, so growth over the years does not inflate the
# later years the way it does in the period-average method.
# The very first value has no previous period: it is used only as the base of the
# second period's ratio, and its own ratio is left empty (the default, as link
# relatives are defined), so period 1 is averaged over its real ratios only.
first_ratio = np.nan
ratios = np.append(first_ratio, series[1:] / series[:-1]).reshape(n_years, periods).T

## Seasonal index

In [ ]:
# Persons' link relative method. No function in pandas, SciPy or statsmodels computes
# it, so the four steps are written out:
# 1. Average each period's ratios across the years.
# 2. Divide by their geometric mean, the average growth per period, so the chain closes:
#    after one full year it returns to where it started.
# 3. Chain the corrected ratios from period 1 (chain index, period 1 = 1).
# 4. Divide by the arithmetic mean of the chain, so the indices average 1. An index of
#    1.29 reads "29% above an average quarter once growth is taken out".
# Two conventions differ from Persons (1919): the ratios are averaged by their mean,
# where he used the median, and the growth is divided out (a geometric correction),
# where he subtracted it in equal steps. Both are in textbooks; they move the indices
# in the second or third decimal. The method assumes a multiplicative season that
# keeps the same shape every year.
mean_ratio = np.nanmean(ratios, axis=1)
geometric = np.exp(np.log(mean_ratio).mean())
corrected = mean_ratio / geometric
chain = np.cumprod(np.append(1.0, corrected[1:]))
arithmetic = chain.mean()
seasonal = chain / arithmetic

# The starting period only renumbers the periods within the year: the first value is
# period start_period, so row p of the table is period (start_period - 1 + p) % periods + 1,
# here and in the adjusted values below.
start_period = 1
period_labels = [(start_period - 1 + p) % periods + 1 for p in range(periods)]
indices = pd.DataFrame(ratios, columns=years, index=pd.Index(period_labels, name="Period"))
indices["Mean ratio to the previous period"] = mean_ratio
indices["Corrected mean"] = corrected
indices["Chain index"] = chain
indices["Seasonal index"] = seasonal
print(indices)
print(f"Geometric mean: {geometric:.10f}    Arithmetic mean: {arithmetic:.10f}")

## Observations and seasonally adjusted values

In [ ]:
# Each value divided by its period's seasonal index: what is left is trend and noise.
adjusted = pd.DataFrame({
    "Year": [years[c] if p == 0 else "" for c in range(n_years) for p in range(periods)],
    "Period": period_labels * n_years,
    "Observed": series,
    "Seasonally adjusted": series / np.tile(seasonal, n_years),
})
print()
print(adjusted.to_string(index=False))

## What the result says

The link relative method divides each quarter's revenue by the quarter before it. These ratios do not depend on how large the business is, so growth does not distort the seasonal pattern. The method averages the ratios for each quarter, chains them together and corrects them for growth.

The seasonal index, where 1 is an average quarter, is 1.289 for the third quarter (summer) and 1.095 for the second. It is 0.813 for the first quarter and 0.803 for the fourth. So summer sells clearly more than an average quarter, and the first and last quarters about a fifth less.

Dividing revenue by its index gives the seasonally adjusted value. It rises from 975 in the first quarter of 2020 to 1553 in the last quarter of 2024, in \$000. With the season removed, revenue grew steadily over the five years.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.